# GAN Training — Colab Runner

Trains a PyTorch vanilla GAN on MNIST, saves checkpoints + figures, and
bundles everything into a downloadable zip.

**Before you run:**  `Runtime → Change runtime type → GPU` (any T4 / L4 / A100
works). With a T4 the full 400-epoch run is ~15–25 minutes.

**What you get at the end:** a `gan_outputs.zip` auto-downloaded to your
machine containing the trained weights (`generator_final.pt`), milestone
sample grids, loss curves, and `eval_stats.json`. Drop those files into the
companion Streamlit app's `gan_outputs/` folder to host the demo.

## 0. Environment check

In [ ]:
import torch, sys
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available(), "—", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import csv
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from tqdm.auto import tqdm

# ---- hyperparameters ----
SEED            = 42
NOISE_DIM       = 100
IMG_DIM         = 28 * 28                 # 784
BATCH_SIZE      = 128
EPOCHS          = 400
LR              = 2e-4
BETA_1, BETA_2  = 0.5, 0.999
SAMPLE_EPOCHS   = [1, 30, 100, 400]
OUT_DIR         = Path("./gan_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

## 2. Generator

Three Dense blocks (256 → 512 → 1024), each with `LeakyReLU(0.2)` and
`BatchNorm1d`. Final `tanh` bounds pixels to `[-1, 1]` to match the
preprocessed real data.

> **BN momentum note.** Keras `momentum=0.8` ⇔ PyTorch `momentum=0.2`.
> The convention is flipped between frameworks.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim: int = NOISE_DIM, img_dim: int = IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, momentum=0.2),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, momentum=0.2),

            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, momentum=0.2),

            nn.Linear(1024, img_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

## 3. Discriminator

Mirror of the generator: three Dense blocks (1024 → 512 → 256) with
`LeakyReLU(0.2)`, then a sigmoid output giving `P(real)`. Batch-norm is
deliberately omitted from D — it destabilises GAN training.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_dim: int = IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## 4. Helpers — sample grids and MNIST loader

In [ ]:
def save_sample_grid(generator, path, epoch, fixed_noise):
    '''Save a 10x10 grid of images generated from `fixed_noise`.'''
    was_training = generator.training
    generator.eval()
    with torch.no_grad():
        imgs = generator(fixed_noise).cpu().numpy()
    imgs = imgs.reshape(-1, 28, 28)
    imgs = (imgs + 1.0) / 2.0  # [-1, 1] -> [0, 1]

    fig, axes = plt.subplots(10, 10, figsize=(10, 10))
    fig.suptitle(f"Generator output — epoch {epoch}", fontsize=14)
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i], cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    plt.tight_layout(); plt.subplots_adjust(top=0.95)
    plt.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    if was_training:
        generator.train()


def load_mnist_tensor(device):
    '''Download MNIST once and return the whole training set as a single
    (60000, 784) tensor on `device`, normalised to [-1, 1].

    Loading to GPU once and indexing into it per-batch is much faster than
    using a DataLoader for a dataset this small, and eliminates the
    num_workers>0 multiprocessing cleanup errors that Colab notebooks hit.'''
    ds = datasets.MNIST(root="./data", train=True, download=True)
    X = ds.data.float().view(-1, 784)
    X = (X - 127.5) / 127.5
    return X.to(device)

## 5. Build models and download MNIST

In [ ]:
X = load_mnist_tensor(DEVICE)                      # (60000, 784) on GPU
STEPS_PER_EPOCH = X.size(0) // BATCH_SIZE          # 468
print(f"MNIST: {X.size(0):,} images loaded to {DEVICE}  ({X.element_size()*X.numel()/1e6:.0f} MB)")
print(f"Steps per epoch: {STEPS_PER_EPOCH}")

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
print(f"Generator params    : {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")

# Two separate optimisers — each owns its own network's parameters.
# No Keras `trainable = False` gotchas, and no DataLoader workers.
g_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA_1, BETA_2))
d_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA_1, BETA_2))
bce   = nn.BCELoss()

# Reused across milestones so we can see *the same latent inputs* evolve
fixed_noise = torch.randn(100, NOISE_DIM, device=DEVICE)

## 6. Untrained generator output (sanity check)

In [ ]:
save_sample_grid(G, OUT_DIR / "samples_epoch_000_untrained.png",
                 epoch="0 (untrained)", fixed_noise=fixed_noise)

from IPython.display import Image, display
display(Image(str(OUT_DIR / "samples_epoch_000_untrained.png")))

## 7. Train for 400 epochs

This is the long cell. On a Colab T4 GPU, expect roughly 15–25 minutes total.

Milestone snapshots are saved at epochs 1, 30, 100, and 400 — both the
sample grid (PNG) and the model weights (`.pt`).

In [ ]:
history = {"epoch": [], "d_loss": [], "d_acc": [], "g_loss": []}
t0 = time.time()

real_labels = torch.full((BATCH_SIZE, 1), 1.0, device=DEVICE)
fake_labels = torch.full((BATCH_SIZE, 1), 0.0, device=DEVICE)

for epoch in range(1, EPOCHS + 1):
    G.train(); D.train()
    d_losses, d_accs, g_losses = [], [], []

    show_bar = (epoch in SAMPLE_EPOCHS) or (epoch % 25 == 0) or (epoch == 1)
    iterator = range(STEPS_PER_EPOCH)
    if show_bar:
        iterator = tqdm(iterator, desc=f"epoch {epoch:3d}", leave=False)

    for _ in iterator:
        # Sample real batch — random index into the GPU-resident tensor
        idx = torch.randint(0, X.size(0), (BATCH_SIZE,), device=DEVICE)
        real_imgs = X[idx]

        # ---- train D ----
        d_opt.zero_grad()
        d_real = D(real_imgs)
        loss_d_real = bce(d_real, real_labels)

        noise = torch.randn(BATCH_SIZE, NOISE_DIM, device=DEVICE)
        fake_imgs = G(noise)
        d_fake = D(fake_imgs.detach())          # detach = no gradient into G
        loss_d_fake = bce(d_fake, fake_labels)

        d_loss = loss_d_real + loss_d_fake
        d_loss.backward()
        d_opt.step()

        with torch.no_grad():
            acc_real = (d_real > 0.5).float().mean().item()
            acc_fake = (d_fake < 0.5).float().mean().item()
            d_acc = 0.5 * (acc_real + acc_fake)

        # ---- train G ----
        g_opt.zero_grad()
        noise = torch.randn(BATCH_SIZE, NOISE_DIM, device=DEVICE)
        fake_imgs = G(noise)
        d_on_fake = D(fake_imgs)                # no detach — gradient flows to G
        g_loss = bce(d_on_fake, real_labels)    # trick: claim fakes are real
        g_loss.backward()
        g_opt.step()

        d_losses.append(d_loss.item() / 2)
        d_accs.append(d_acc)
        g_losses.append(g_loss.item())

    history["epoch"].append(epoch)
    history["d_loss"].append(float(np.mean(d_losses)))
    history["d_acc"].append(float(np.mean(d_accs)))
    history["g_loss"].append(float(np.mean(g_losses)))

    if epoch in SAMPLE_EPOCHS:
        save_sample_grid(G, OUT_DIR / f"samples_epoch_{epoch:03d}.png", epoch=epoch, fixed_noise=fixed_noise)
        torch.save(G.state_dict(), OUT_DIR / f"generator_epoch{epoch:03d}.pt")
        torch.save(D.state_dict(), OUT_DIR / f"discriminator_epoch{epoch:03d}.pt")
        elapsed = (time.time() - t0) / 60.0
        print(f"[epoch {epoch:3d}] d_loss={history['d_loss'][-1]:.4f}  "
              f"d_acc={history['d_acc'][-1]:.3f}  g_loss={history['g_loss'][-1]:.4f}  "
              f"elapsed={elapsed:.1f} min")
    elif epoch % 25 == 0:
        elapsed = (time.time() - t0) / 60.0
        print(f"  epoch {epoch:3d}  d_loss={history['d_loss'][-1]:.4f}  "
              f"d_acc={history['d_acc'][-1]:.3f}  g_loss={history['g_loss'][-1]:.4f}  "
              f"({elapsed:.1f} min)")

total = (time.time() - t0) / 60.0
print(f"\nFinished in {total:.1f} min")

# Final checkpoint used by the Streamlit app
torch.save(G.state_dict(), OUT_DIR / "generator_final.pt")
torch.save(D.state_dict(), OUT_DIR / "discriminator_final.pt")

## 8. Loss curves and training history

In [ ]:
with open(OUT_DIR / "history.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["epoch", "d_loss", "d_acc", "g_loss"])
    for e, dl, da, gl in zip(history["epoch"], history["d_loss"], history["d_acc"], history["g_loss"]):
        w.writerow([e, dl, da, gl])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history["epoch"], history["d_loss"], color="#B85042", linewidth=1.2, label="Discriminator loss")
axes[0].plot(history["epoch"], history["g_loss"], color="#065A82", linewidth=1.2, label="Generator loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("GAN training losses")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(history["epoch"], history["d_acc"], color="#2C5F2D", linewidth=1.2, label="Discriminator accuracy")
axes[1].axhline(0.5, color="grey", linestyle="--", alpha=0.7, label="Ideal equilibrium (0.5)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05); axes[1].set_title("Discriminator accuracy")
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "loss_curves.png", dpi=110, bbox_inches="tight")
plt.show()

## 9. Quantitative evaluation

Score 5 000 real MNIST images and 5 000 freshly generated fakes, and look at
the distribution of the discriminator's output. If G has learned, the two
histograms should overlap substantially.

In [ ]:
G.eval(); D.eval()
N_EVAL = 5000

with torch.no_grad():
    idx = torch.randperm(X.size(0), device=DEVICE)[:N_EVAL]
    real_sample = X[idx]
    noise_eval  = torch.randn(N_EVAL, NOISE_DIM, device=DEVICE)
    fake_sample = G(noise_eval)

    d_real = D(real_sample).cpu().numpy().flatten()
    d_fake = D(fake_sample).cpu().numpy().flatten()

stats = {
    "device": str(DEVICE),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "total_minutes": total,
    "final_d_loss": history["d_loss"][-1],
    "final_g_loss": history["g_loss"][-1],
    "final_d_acc":  history["d_acc"][-1],
    "mean_disc_score_on_real":  float(d_real.mean()),
    "mean_disc_score_on_fake":  float(d_fake.mean()),
    "pct_fake_classified_real": float((d_fake > 0.5).mean()),
    "pct_real_classified_real": float((d_real > 0.5).mean()),
}
with open(OUT_DIR / "eval_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(d_real, bins=40, alpha=0.7, color="#2C5F2D", label="Real MNIST")
ax.hist(d_fake, bins=40, alpha=0.7, color="#B85042", label="Generator output")
ax.axvline(0.5, color="black", linestyle="--", alpha=0.5, label="Decision threshold")
ax.set_xlabel("Discriminator score  —  P(image is real)")
ax.set_ylabel(f"Count (of {N_EVAL})")
ax.set_title("Discriminator confidence: real vs. generated")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "disc_confidence.png", dpi=110, bbox_inches="tight")
plt.show()

print("\nQuantitative results:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## 10. Display all milestone sample grids side-by-side

In [ ]:
from IPython.display import Image, display
for ep in SAMPLE_EPOCHS:
    print(f"Epoch {ep}:")
    display(Image(str(OUT_DIR / f"samples_epoch_{ep:03d}.png")))

## 11. Download everything

Zips `gan_outputs/` and auto-downloads it. Drop the contents into the
Streamlit app's `gan_outputs/` folder to light up the live demo.

In [ ]:
import shutil
zip_path = shutil.make_archive("gan_outputs", "zip", ".", "gan_outputs")
print("Bundle created:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")

try:
    # Colab: trigger a browser download
    from google.colab import files
    files.download(zip_path)
except ImportError:
    # Not on Colab — just print the path
    print("Running outside Colab. Grab the zip from:", zip_path)